# 02 - MODEL TRAINING AND EVALUATION

Linear regression with gradient descent, implemented in pure Python.  
Input: `data/processed/mobile_phone_price_clean.csv`.

In [1]:
import pandas as pd
import random
import math

print("libraries imported successfully")

libraries imported successfully


# LOAD THE CLEANED DATA

In [2]:
df = pd.read_csv("../data/processed/mobile_phone_price_clean.csv")

df.head()

,Brand,Model,Storage,RAM,Screen_size,Num_camera,Cam_main,Cam_ultra,Cam_telephoto,Cam_macro,Battery_cap,Price
0,Apple,iPhone 13 Pro,128,6,6.10,3,12.0,12.0,12.0,0.0,3095,999
1,Samsung,Galaxy S21 Ultra,256,12,6.80,4,108.0,10.0,10.0,0.0,5000,1199
2,OnePlus,9 Pro,128,8,6.70,4,48.0,50.0,8.0,2.0,4500,899
3,Xiaomi,Redmi Note 10 Pro,128,6,6.67,4,64.0,8.0,0.0,5.0,5020,279
4,Google,Pixel 6,128,8,6.40,2,50.0,12.2,0.0,0.0,4614,799


# ENCODE CATEGORICAL COLUMNS

In [3]:
df=df.drop(columns=["Model"])
df_encoded = pd.get_dummies(df,columns=['Brand'],drop_first=True)
df_encoded.shape

(358, 25)

# DEFINE FEATURES (X) AND TARGET (y)

In [4]:
X= df_encoded.drop(columns=['Price'])

y = df_encoded["Price"]

print("Features (X):")
display(X.head())

print("\nTarget (y):")
display(y.tail())
print('Target shpe: ',y.shape)

Features (X):


,Storage,RAM,Screen_size,Num_camera,Cam_main,Cam_ultra,Cam_telephoto,Cam_macro,Battery_cap,Brand_Asus,...,Brand_LG,Brand_Motorola,Brand_Nokia,Brand_OnePlus,Brand_Oppo,Brand_Realme,Brand_Samsung,Brand_Sony,Brand_Vivo,Brand_Xiaomi
0,128,6,6.10,3,12.0,12.0,12.0,0.0,3095,False,...,False,False,False,False,False,False,False,False,False,False
1,256,12,6.80,4,108.0,10.0,10.0,0.0,5000,False,...,False,False,False,False,False,False,True,False,False,False
2,128,8,6.70,4,48.0,50.0,8.0,2.0,4500,False,...,False,False,False,True,False,False,False,False,False,False
3,128,6,6.67,4,64.0,8.0,0.0,5.0,5020,False,...,False,False,False,False,False,False,False,False,False,True
4,128,8,6.40,2,50.0,12.2,0.0,0.0,4614,False,...,False,False,False,False,False,False,False,False,False,False



Target (y):


353     329
354    1049
355     349
356    1099
357     429
Name: Price, dtype: int64

Target shpe:  (358,)


# SPLIT DATA INTO TRAINING AND TESTING SETS

In [5]:
def train_test_split(X,y,test_fac=0.20,seed=42):
    n=len(X)
    rng=random.Random(seed)
    indices=list(range(n))
    rng.shuffle(indices)

    n_train=int(n*(1-test_fac))
    train_idx,test_idx=indices[:n_train],indices[n_train:]

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    return(X_train, X_test, y_train, y_test)

X_train, X_test, y_train, y_test = train_test_split( X, y, test_fac=0.20,seed=42)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 286
Testing samples: 72


# SCALE THE FEATURES

In [6]:
def scaler(X_train, X_test):
    train_mean = X_train.mean(axis=0)
    train_std = X_train.std(axis=0, ddof=0).replace(0, 1e-9)  
    
    X_train_scaled = (X_train - train_mean) / train_std
    X_test_scaled = (X_test - train_mean) / train_std
    
    return X_train_scaled, X_test_scaled
    
X_train_scaled,X_test_scaled=scaler(X_train,X_test)
print("Features standardized successfully.")

Features standardized successfully.


# TRAIN A MACHINE-LEARNING MODEL

In [8]:
def linear_model(X_train_scaled,y_train,lr=0.01, epochs=1000):
    n_samples=len(X_train_scaled)
    n_features=len(X_train_scaled[1])

    w=[0.0]*n_features
    b=0.0

    for epoch in range(epochs):
        #prediction
        y_pred=[]
        for i in range(n_samples):
            pred=0.0
            for j in range(n_features):
                pred+=X_train_scaled[i][j]*w[j]
            y_pred.append(pred+b)
            
        #error
        errors=[]
        for i in range(n_samples):
            error=y_pred[i]-y_train[i]
            errors.append(error)
                
        #loss
        dw=[0.0] * n_features
        for i in range(n_features):
            for j in range(n_samples):
                dw[i]+=(2 / n_samples) * errors[j] * X_train_scaled[j][i]
                                              
        db = (2 / n_samples) * sum(errors)

        #update weight and bais
        for j in range(n_features):
            w[j] -= lr * dw[j]
        b -= lr * db

        if (epoch + 1) % 100 == 0:
            mse = sum(err ** 2 for err in errors) / n_samples
            print(f"Epoch {epoch + 1}/{epochs} - Loss (MSE): {mse:.2f}")
    final_mse=sum(err ** 2 for err in errors) / n_samples
    return w, b, final_mse

def predict(X, w, b):
    n_samples=len(X)
    n_features=len(w)
    predictions=[]
    for i in range(n_samples):
        pred = sum(X[i][j] * w[j] for j in range(n_features)) + b
        predictions.append(pred)
    return predictions

    
X_train_list = X_train_scaled.values.tolist()
y_train_list = y_train.values.tolist()
#y_train_log_list = y_train_log.values.tolist()

w, b,MSE = linear_model(X_train_list, y_train_list, lr=0.01, epochs=10000)


print(f'\nweight: {w}')
print(f"learned_bais: {b:.2f}")
print(f"Loss: {MSE:.2f}")

Epoch 100/10000 - Loss (MSE): 17126.27
Epoch 200/10000 - Loss (MSE): 12710.60
Epoch 300/10000 - Loss (MSE): 11867.97
Epoch 400/10000 - Loss (MSE): 11318.75
Epoch 500/10000 - Loss (MSE): 10899.94
Epoch 600/10000 - Loss (MSE): 10561.07
Epoch 700/10000 - Loss (MSE): 10277.04
Epoch 800/10000 - Loss (MSE): 10034.00
Epoch 900/10000 - Loss (MSE): 9823.52
Epoch 1000/10000 - Loss (MSE): 9640.00
Epoch 1100/10000 - Loss (MSE): 9479.37
Epoch 1200/10000 - Loss (MSE): 9338.46
Epoch 1300/10000 - Loss (MSE): 9214.70
Epoch 1400/10000 - Loss (MSE): 9105.93
Epoch 1500/10000 - Loss (MSE): 9010.29
Epoch 1600/10000 - Loss (MSE): 8926.17
Epoch 1700/10000 - Loss (MSE): 8852.18
Epoch 1800/10000 - Loss (MSE): 8787.08
Epoch 1900/10000 - Loss (MSE): 8729.82
Epoch 2000/10000 - Loss (MSE): 8679.43
Epoch 2100/10000 - Loss (MSE): 8635.10
Epoch 2200/10000 - Loss (MSE): 8596.10
Epoch 2300/10000 - Loss (MSE): 8561.79
Epoch 2400/10000 - Loss (MSE): 8531.60
Epoch 2500/10000 - Loss (MSE): 8505.03
Epoch 2600/10000 - Loss (M

# EVALUATE THE MODEL

In [9]:
def evaluate_model(X_test, y_test, w, b,):

    y_pred = predict(X_test, w, b)
    y_actual = y_test

    n_samples = len(y_actual)
    mae = sum(abs(y_actual[i] - y_pred[i]) for i in range(n_samples)) / n_samples
    mse = sum((y_actual[i] - y_pred[i]) ** 2 for i in range(n_samples)) / n_samples
    rmse = math.sqrt(mse)
    
    mean_actual = sum(y_actual) / n_samples
    ss_tot = sum((y_actual[i] - mean_actual) ** 2 for i in range(n_samples))
    ss_res = sum((y_actual[i] - y_pred[i]) ** 2 for i in range(n_samples))
    r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0.0

    print("=" * 40)
    print("        MODEL EVALUATION METRICS        ")
    print("=" * 40)
    print(f"Mean Absolute Error (MAE)     : {mae:.2f}")
    print(f"Mean Squared Error (MSE)     : {mse:.2f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
    print(f"R-squared Score (R²)          : {r2:.4f}")
    print("-" * 40)
    
    print("\nFirst 5 Actual vs. Predicted Values:")
    for i in range(min(5, n_samples)):
        print(f"Actual: {y_actual[i]:.2f} | Predicted: {y_pred[i]:.2f}")

    return {"mae": mae, "mse": mse, "rmse": rmse, "r2": r2}

    
X_test_list = X_test_scaled.values.tolist()
y_test_list=y_test.values.tolist()
metrics = evaluate_model(X_test_list, y_test_list, w, b)

        MODEL EVALUATION METRICS        
Mean Absolute Error (MAE)     : 96.43
Mean Squared Error (MSE)     : 20070.76
Root Mean Squared Error (RMSE): 141.67
R-squared Score (R²)          : 0.8379
----------------------------------------

First 5 Actual vs. Predicted Values:
Actual: 169.00 | Predicted: 138.57
Actual: 199.00 | Predicted: 363.20
Actual: 269.00 | Predicted: 252.75
Actual: 1299.00 | Predicted: 927.31
Actual: 219.00 | Predicted: 299.64
